# Amazon ML Challenge 2026: Business Entity Resolution, FULL-DATA run (Colab + Google Drive)

**Pipeline:** frozen `BAAI/bge-m3` embeddings (no fine-tuning) → exact per-country GPU retrieval
(country hard filter) → fine-tuned `BAAI/bge-reranker-v2-m3` cross-encoder → threshold tuned for
macro F0.5 on a held-out validation split → `matching_results.tsv` + `candidate_pairs.tsv`.

**How to run:** Runtime → Change runtime type → **L4 GPU** (T4 also works, slower), then
**Runtime → Run all**. That's it. Every stage saves its output to Google Drive (`WORK` folder) and
skips itself if that output already exists. If Colab disconnects, reconnect and **Run all** again:
it resumes where it stopped, including mid-way through embedding shards and cross-encoder training
(checkpointed to Drive every `CKPT_MINUTES`).

| Stage | What it does | Output on Drive (`WORK/…`) |
|---|---|---|
| 1 | Mount Drive | |
| 2 | Config + environment check | |
| 3 | Get raw data (already on Drive, or Kaggle API) | `raw_data/` |
| 4 | Load + normalize all sources (train + test) | `prep/*.parquet` |
| 5 | Ground truth → index arrays, country hard-filter check | `prep/train_gt.npz` |
| 6 | Encode **every** record with frozen bge-m3 (fp16, 1024-d) | `emb/<split>_<src>/shard_*.npy` |
| 7 | Train/val split | `split.npz` |
| 8 | Exact top-K retrieval per country, per source (S2 and S3 separately) | `cand/*.npz` |
| 9 | Recall@K report (blocking ceiling) | |
| 10 | Cross-encoder training pairs (positives + retrieved hard negatives) | `ce_data/pairs.parquet` |
| 11 | Fine-tune bge-reranker-v2-m3 (resumable, OOM-safe) | `ce_ckpt/last.pt`, `ce_final/` |
| 12 | Score validation candidates, tune thresholds for macro F0.5 | `thresholds.json` |
| 13 | Test set: retrieve, write candidate_pairs, score, write matching_results, validate | `submission/` |

**Why the old notebook ran out of memory:** it fine-tuned bge-m3 with `MultipleNegativesRankingLoss` at
batch 128. That loss backprops through 256 sequences at once, which is what filled the 22 GB GPU. This
version does no bi-encoder training at all: bge-m3 is only used for inference, in fp16 and under
`torch.inference_mode()`. Every GPU loop here (encoding, retrieval, training, scoring) also catches
CUDA OOM and retries with a smaller batch instead of crashing.

In [ ]:
# ---- Cell 1: mount Google Drive ----
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---- Cell 2: config + environment ----
import os
os.environ.setdefault("HF_HOME", "/content/hf_cache")          # model downloads: fast local disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import gc, csv, json, math, time, random, shutil, subprocess, sys, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
transformers.logging.set_verbosity_error()

try:   # optional: silences the "unauthenticated requests to the HF Hub" warning if you add an HF_TOKEN secret
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------- paths ----------------
DRIVE_DIR = "/content/drive/MyDrive/amazon_ml_challenge"
TRAIN_DIR = f"{DRIVE_DIR}/raw_data"          # train_source1/2/3.tsv + train_ground_truth.tsv
TEST_DIR = f"{DRIVE_DIR}/raw_data/test"      # test_source1/2/3.tsv  (upload these here)
WORK = f"{DRIVE_DIR}/full_v2"                # every artifact of THIS notebook (separate from old runs)
LOCAL_TMP = "/content/tmp"                   # scratch for checkpoint staging
KAGGLE_TRAIN_SLUG = "dhruvgupta1606/amazon-train-1"   # only used if the train TSVs aren't on Drive yet

# ---------------- models ----------------
EMB_MODEL = "BAAI/bge-m3"                    # frozen, never trained here (MIT, 568M)
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"   # fine-tuned below (Apache-2.0, 568M)

# ---------------- embeddings ----------------
EMB_MAX_LEN = 128            # tokens; the longest record in the audit is ~46 words
EMB_BATCH = 512              # starting batch; auto-halves on OOM
EMB_SHARD_ROWS = 250_000     # rows per saved shard (= resume granularity)

# ---------------- retrieval / blocking ----------------
K_PER_SOURCE = 25            # top-K from S2 AND top-K from S3, same country only
RERANK_K = 20                # best RERANK_K of those 2*K_PER_SOURCE go to the cross-encoder
                             # (= candidate_pairs.tsv). Ground truth max is 11 matches per entity.
QUERY_BLOCK = 4096           # queries per GPU matmul block (auto-halves on OOM)

# ---------------- split ----------------
VAL_FRACTION = 0.05          # of train S1 entities, held out
N_VAL_EVAL = 50_000          # val entities actually scored for threshold tuning (enough for a stable F0.5)

# ---------------- cross-encoder training ----------------
N_CE_TRAIN_ENTITIES = 150_000   # train S1 entities used to build CE pairs (~11 pairs each ≈ 1.7M pairs)
NEG_HARD = 4                  # hardest retrieved non-matches per entity
NEG_RANDOM = 4                # + random other retrieved non-matches per entity
CE_MAX_LEN = 192              # tokens for the concatenated (S1, candidate) pair
CE_EPOCHS = 1
CE_LR = 2e-5
CE_WARMUP_FRAC = 0.05
CE_WEIGHT_DECAY = 0.01
CKPT_MINUTES = 30             # full training checkpoint (model + optimizer) to Drive this often
SCORE_BATCH = 512             # cross-encoder inference batch; auto-halves on OOM
SCORE_CHUNK = 200_000         # pairs per saved score file (= scoring resume granularity)

# ---------------- device ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    GPU_GB = _p.total_memory / 1e9
    # bf16 only on Ampere+ (L4/A100). T4 reports "bf16 supported" via slow emulation, so check capability.
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    CE_BATCH = 32 if GPU_GB >= 20 else 16
    print(f"GPU: {_p.name} ({GPU_GB:.1f} GB) | AMP dtype: {AMP_DTYPE} | CE batch: {CE_BATCH}")
else:
    GPU_GB, AMP_DTYPE, CE_BATCH = 0.0, torch.float32, 16
    print("WARNING: no GPU found. Runtime → Change runtime type → L4/T4 GPU, then Run all again.")
INFER_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} "
      f"| pandas {pd.__version__}")

P = {k: f"{WORK}/{k}" for k in ("prep", "emb", "cand", "ce_data", "ce_ckpt", "ce_final", "scores", "submission")}
for _d in list(P.values()) + [LOCAL_TMP, TRAIN_DIR]:
    os.makedirs(_d, exist_ok=True)
print("Artifacts folder:", WORK)

# ---------------- small helpers ----------------
def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)

def _tmp(path):
    return f"{path}.tmp{os.getpid()}"

def save_npy(path, arr):          # atomic: a disconnect mid-write never leaves a half file behind
    tmp = _tmp(path)
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)

def save_npz(path, **arrays):
    tmp = _tmp(path)
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)

def load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def save_parquet(df, path):
    tmp = _tmp(path)
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def save_json(obj, path):
    tmp = _tmp(path)
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def free_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

## 3. Raw data

Train files are expected at `MyDrive/amazon_ml_challenge/raw_data/`. If they're missing, the cell
downloads them once with the Kaggle API (Colab Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`, with
"Notebook access" on) and keeps them on Drive for good.

**Test files:** upload `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv` to
`MyDrive/amazon_ml_challenge/raw_data/test/`. If they aren't there, everything else still runs and the
test stage (13) prints a note and skips. Add the files later and **Run all** again: all finished stages
skip in seconds.

In [ ]:
# ---- Cell 3: raw data ----
TRAIN_FILES = ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]
TEST_FILES = ["test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]

def _missing(dir_, files):
    return [f for f in files if not os.path.exists(f"{dir_}/{f}")]

if _missing(TRAIN_DIR, TRAIN_FILES):
    log(f"Train files missing on Drive ({_missing(TRAIN_DIR, TRAIN_FILES)}), downloading from Kaggle...")
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception as e:
        raise RuntimeError(
            "Train TSVs are not in MyDrive/amazon_ml_challenge/raw_data/ and Kaggle secrets are not set. "
            "Either upload the 4 train files there, or add KAGGLE_USERNAME / KAGGLE_KEY as Colab secrets.") from e
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    dl = f"{LOCAL_TMP}/kaggle_dl"
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_TRAIN_SLUG, "-p", dl, "--unzip"], check=True)
    for root, _, files in os.walk(dl):          # zips sometimes nest the files one folder deeper
        for fn in files:
            if fn in TRAIN_FILES:
                shutil.copy(f"{root}/{fn}", f"{TRAIN_DIR}/{fn}")
            elif fn in TEST_FILES:
                os.makedirs(TEST_DIR, exist_ok=True)
                shutil.copy(f"{root}/{fn}", f"{TEST_DIR}/{fn}")
    assert not _missing(TRAIN_DIR, TRAIN_FILES), f"still missing after download: {_missing(TRAIN_DIR, TRAIN_FILES)}"

HAVE_TEST = not _missing(TEST_DIR, TEST_FILES)
SPLITS = ["train", "test"] if HAVE_TEST else ["train"]
print("Train files:", sorted(os.listdir(TRAIN_DIR)))
print("Test files present:", HAVE_TEST, "" if HAVE_TEST else f"(upload {TEST_FILES} to {TEST_DIR})")

## 4. Load + normalize

Unicode-safe normalization: NFKC, lowercase, `&`→`and`. Punctuation is stripped by Unicode category
(keep letters, marks, numbers), so Devanagari combining marks survive. Legal suffixes and street
abbreviations are canonicalized. Each record becomes one text:
`name: <name> | address: <address>` (address part omitted when missing, ~3.3% of S2/S3).

Files are read with `keep_default_na=False` so a business literally named "NA" or "None" is not
turned into a missing value. The row count is checked against the raw line count to catch
quote-parsing problems.

In [ ]:
# ---- Cell 4: load + normalize (cached to Drive) ----
try:
    import regex as _regex
    _PUNCT = _regex.compile(r"[^\p{L}\p{M}\p{N}\s]+")
    def _strip_punct(s):
        return _PUNCT.sub(" ", s)
except ImportError:
    def _strip_punct(s):
        return "".join(ch if (unicodedata.category(ch)[0] in "LMN" or ch.isspace()) else " " for ch in s)

SUFFIX_MAP = {"pvt": "private", "ltd": "limited", "inc": "incorporated", "corp": "corporation",
              "co": "company"}
ADDR_ABBR_MAP = {"rd": "road", "st": "street", "ave": "avenue", "blvd": "boulevard", "dr": "drive",
                 "ln": "lane", "apt": "apartment", "ste": "suite", "hwy": "highway", "ct": "court",
                 "pl": "place", "sq": "square"}

def norm_text(s):
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s).lower().replace("&", " and ")
    return " ".join(_strip_punct(s).split())

def norm_name(s):
    return " ".join(SUFFIX_MAP.get(t, t) for t in norm_text(s).split())

def norm_addr(s):
    return " ".join(ADDR_ABBR_MAP.get(t, t) for t in norm_text(s).split())

def make_text(name, addr):
    return f"name: {name} | address: {addr}" if addr else f"name: {name}"

def _count_data_lines(path):
    n, last = 0, b"\n"
    with open(path, "rb") as f:
        while True:
            b = f.read(1 << 24)
            if not b:
                break
            n += b.count(b"\n")
            last = b[-1:]
    return n + (0 if last == b"\n" else 1) - 1      # minus header

def read_tsv(path):
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_values=[""])
    expected = _count_data_lines(path)
    if len(df) != expected:
        log(f"  {os.path.basename(path)}: {len(df):,} rows parsed vs {expected:,} lines, trying quoting off")
        try:
            df2 = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_values=[""],
                              quoting=csv.QUOTE_NONE)
            if abs(len(df2) - expected) < abs(len(df) - expected) and list(df2.columns) == list(df.columns):
                df = df2
                log(f"  -> using quoting-off parse ({len(df):,} rows)")
        except Exception as e:
            log(f"  -> quoting-off parse failed ({type(e).__name__}), keeping the default parse")
    return df.fillna("")

def prep_source(split, src):
    out = f"{P['prep']}/{split}_{src}.parquet"
    if os.path.exists(out):
        return pd.read_parquet(out)
    path = f"{TRAIN_DIR if split == 'train' else TEST_DIR}/{split}_source{src[1]}.tsv"
    t0 = time.time()
    df = read_tsv(path)
    for c in ("entity_id", "business_name", "business_address", "country"):
        assert c in df.columns, f"{path} is missing column {c}: {list(df.columns)}"
    ids = df["entity_id"].str.strip()
    assert ids.is_unique, f"duplicate entity_id in {path}"
    names = [norm_name(x) for x in df["business_name"].values]
    addrs = [norm_addr(x) for x in df["business_address"].values]
    res = pd.DataFrame({
        "entity_id": ids.values,
        "country": df["country"].str.strip().values,
        "text": [make_text(n, a) for n, a in zip(names, addrs)],
    })
    save_parquet(res, out)
    log(f"  prepped {split}_{src}: {len(res):,} rows in {time.time()-t0:.0f}s")
    return res

DATA = {}
for split in SPLITS:
    s1, s2, s3 = (prep_source(split, s) for s in ("s1", "s2", "s3"))
    countries = sorted(set(s1["country"]) | set(s2["country"]) | set(s3["country"]))
    cmap = {c: i for i, c in enumerate(countries)}
    DATA[split] = {
        "s1_ids": s1["entity_id"].values, "s1_text": s1["text"].values,
        "s1_cc": s1["country"].map(cmap).values.astype(np.int16),
        # corpus = S2 rows followed by S3 rows; "corpus index" below always means this concatenation
        "c_ids": np.concatenate([s2["entity_id"].values, s3["entity_id"].values]),
        "c_text": np.concatenate([s2["text"].values, s3["text"].values]),
        "c_cc": np.concatenate([s2["country"].map(cmap).values, s3["country"].map(cmap).values]).astype(np.int16),
        "n_s2": len(s2), "n_s3": len(s3), "countries": countries,
    }
    del s1, s2, s3
    d = DATA[split]
    print(f"{split}: S1={len(d['s1_ids']):,}  S2={d['n_s2']:,}  S3={d['n_s3']:,}  countries={countries}")
    for c in countries:
        print(f"   {c:>10}: S1={int((d['s1_cc'] == cmap[c]).sum()):,}  corpus={int((d['c_cc'] == cmap[c]).sum()):,}")
gc.collect()
print("\nexample:", DATA["train"]["s1_text"][0])

## 5. Ground truth + country hard filter check

Retrieval only compares records with the **same country label** (hard filter). Country labels come from
the data (an open set), so France in the test set is handled like any other country. This cell checks,
on the real ground truth, that no true match crosses countries; the printed number is the recall you
would lose to the filter.

In [ ]:
# ---- Cell 5: ground truth as index arrays + country filter check ----
gt_path = f"{P['prep']}/train_gt.npz"
TR = DATA["train"]
if os.path.exists(gt_path):
    GT = load_npz(gt_path)
else:
    gt = read_tsv(f"{TRAIN_DIR}/train_ground_truth.tsv")
    s1_row = pd.Index(TR["s1_ids"]).get_indexer(gt["source1_entity_id"].str.strip())
    assert (s1_row >= 0).all(), "ground truth has S1 ids not in train_source1"
    ex = pd.DataFrame({"r": s1_row, "m": gt["matched_entity_ids"].str.split(",")}).explode("m")
    ex["m"] = ex["m"].fillna("").str.strip()
    ex = ex[ex["m"] != ""]
    c_idx = pd.Index(TR["c_ids"]).get_indexer(ex["m"])
    if (c_idx < 0).any():
        log(f"WARNING: {(c_idx < 0).sum():,} GT ids not found in S2/S3, ignored")
    keep = c_idx >= 0
    GT = {"s1": ex["r"].values[keep].astype(np.int64), "c": c_idx[keep].astype(np.int64)}
    save_npz(gt_path, **GT)
    del gt, ex

N_C_TR = len(TR["c_ids"])
GT_KEYS = np.unique(GT["s1"] * N_C_TR + GT["c"])      # pair key = s1_row * n_corpus + corpus_idx
TRUE_COUNT = np.bincount(GT["s1"], minlength=len(TR["s1_ids"]))

n_edges = len(GT["s1"])
cross = int((TR["s1_cc"][GT["s1"]] != TR["c_cc"][GT["c"]]).sum())
print(f"GT: {n_edges:,} match edges | singletons {int((TRUE_COUNT == 0).sum()):,} "
      f"({(TRUE_COUNT == 0).mean():.2%}) | max matches {TRUE_COUNT.max()}")
print(f"Cross-country GT edges: {cross:,} ({cross / max(n_edges, 1):.4%})")
print("-> country hard filter is SAFE (no recall lost)." if cross == 0 else
      f"-> the hard filter caps edge recall at {1 - cross / n_edges:.4%} (kept anyway: big precision/speed win).")

## 6. Encode every record with frozen bge-m3

Dense bge-m3 embedding = normalized CLS vector of the last hidden state (the model's documented
pooling), fp16, 1024-d, stored as `.npy` shards of 250k rows on Drive (≈0.5 GB per shard, ≈26 GB for
the train set). Within a shard, texts are sorted by length so batches have almost no padding, and the
longest batch runs first so any memory problem shows up immediately (and is handled by halving the
batch). Resume granularity is one shard.

Rough time on an L4: ~2,000 records/s, so ~1.7 h for the 12.5M train records, plus the test set.

In [ ]:
# ---- Cell 6: frozen bge-m3 embeddings for all splits/sources (resumable per shard) ----
def emb_dir(split, src):
    return f"{P['emb']}/{split}_{src}"

def shard_path(split, src, i):
    return f"{emb_dir(split, src)}/shard_{i:04d}.npy"

def n_shards(n):
    return math.ceil(n / EMB_SHARD_ROWS)

@torch.inference_mode()
def encode_texts(texts, tok, model, bs):
    order = np.argsort(np.fromiter((len(t) for t in texts), np.int64, len(texts)), kind="stable")[::-1]
    out = np.empty((len(texts), model.config.hidden_size), np.float16)
    i = 0
    while i < len(order):
        idx = order[i:i + bs]
        try:
            enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=EMB_MAX_LEN,
                      return_tensors="pt").to(DEVICE)
            h = model(**enc).last_hidden_state[:, 0]
            out[idx] = F.normalize(h.float(), dim=-1).to(torch.float16).cpu().numpy()
            i += len(idx)
        except RuntimeError as e:
            if not is_oom(e) or bs == 1:
                raise
            enc = h = None
            free_gpu()
            bs = max(1, bs // 2)
            log(f"  OOM while encoding, batch -> {bs}")
    return out, bs

def split_src_texts(split, src):
    d = DATA[split]
    if src == "s1":
        return d["s1_text"]
    return d["c_text"][:d["n_s2"]] if src == "s2" else d["c_text"][d["n_s2"]:]

jobs = [(sp, src) for sp in SPLITS for src in ("s1", "s2", "s3")]
todo = []
for sp, src in jobs:
    texts = split_src_texts(sp, src)
    os.makedirs(emb_dir(sp, src), exist_ok=True)
    meta_p = f"{emb_dir(sp, src)}/meta.json"
    meta = {"n": int(len(texts)), "model": EMB_MODEL, "shard_rows": EMB_SHARD_ROWS, "max_len": EMB_MAX_LEN}
    if os.path.exists(meta_p):
        old = json.load(open(meta_p))
        assert old == meta, (f"{emb_dir(sp, src)} was built with different settings {old} vs {meta}. "
                             "Delete that folder (or change WORK) to rebuild.")
    else:
        save_json(meta, meta_p)
    todo += [(sp, src, i) for i in range(n_shards(len(texts))) if not os.path.exists(shard_path(sp, src, i))]

if not todo:
    print("All embeddings already on Drive.")
else:
    log(f"{len(todo)} shards to encode (loading {EMB_MODEL}, frozen)")
    emb_tok = AutoTokenizer.from_pretrained(EMB_MODEL)
    emb_model = AutoModel.from_pretrained(EMB_MODEL).to(DEVICE).eval()
    if DEVICE == "cuda":
        emb_model = emb_model.half()
    bs, t_start, done_rows = EMB_BATCH, time.time(), 0
    total_rows = sum(len(split_src_texts(sp, src)[i * EMB_SHARD_ROWS:(i + 1) * EMB_SHARD_ROWS]) for sp, src, i in todo)
    for k, (sp, src, i) in enumerate(todo):
        texts = split_src_texts(sp, src)[i * EMB_SHARD_ROWS:(i + 1) * EMB_SHARD_ROWS]
        E, bs = encode_texts(list(texts), emb_tok, emb_model, bs)
        save_npy(shard_path(sp, src, i), E)
        done_rows += len(texts)
        rate = done_rows / (time.time() - t_start)
        log(f"  [{k+1}/{len(todo)}] {sp}_{src} shard {i}: {len(texts):,} rows | {rate:,.0f} rows/s | "
            f"ETA {(total_rows - done_rows) / rate / 60:.0f} min")
    del emb_model, emb_tok
    free_gpu()
    log("Embeddings done.")

EMB_DIM = int(np.load(shard_path("train", "s1", 0), mmap_mode="r").shape[1])

def load_emb_rows(split, src, rows):
    rows = np.asarray(rows, dtype=np.int64)
    out = np.empty((len(rows), EMB_DIM), np.float16)
    sh = rows // EMB_SHARD_ROWS
    for s in np.unique(sh):
        m = np.nonzero(sh == s)[0]
        E = np.load(shard_path(split, src, int(s)), mmap_mode="r")
        out[m] = E[rows[m] - s * EMB_SHARD_ROWS]
    return out

## 7. Train / validation split (by S1 entity)

In [ ]:
# ---- Cell 7: split ----
split_path = f"{WORK}/split.npz"
if os.path.exists(split_path):
    SPL = load_npz(split_path)
else:
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(TR["s1_ids"]))
    n_val = int(len(perm) * VAL_FRACTION)
    val_rows, train_rows = np.sort(perm[:n_val]), np.sort(perm[n_val:])
    val_eval = np.sort(rng.choice(val_rows, size=min(N_VAL_EVAL, len(val_rows)), replace=False))
    ce_rows = np.sort(rng.choice(train_rows, size=min(N_CE_TRAIN_ENTITIES, len(train_rows)), replace=False))
    SPL = {"val_rows": val_rows, "train_rows": train_rows, "val_eval": val_eval, "ce_rows": ce_rows}
    save_npz(split_path, **SPL)
print(f"train {len(SPL['train_rows']):,} | val {len(SPL['val_rows']):,} | "
      f"val entities scored {len(SPL['val_eval']):,} | CE training entities {len(SPL['ce_rows']):,}")
assert not np.isin(SPL["ce_rows"], SPL["val_rows"]).any()

## 8. Exact per-country retrieval (country hard filter)

No approximate index: with raw fp16 embeddings on Drive, exact inner-product search on the GPU is
fast. It's a matmul per (corpus shard × country × query block), keeping a running top-K per query.
S2 and S3 are searched separately (top `K_PER_SOURCE` each), then merged and sorted by cosine score.
Memory stays flat: one 250k-row shard and one `QUERY_BLOCK × shard` score block on the GPU at a time.

In [ ]:
# ---- Cell 8: retrieval ----
@torch.no_grad()
def retrieve(split, q_rows, K=K_PER_SOURCE):
    d = DATA[split]
    q_rows = np.asarray(q_rows, dtype=np.int64)
    nq = len(q_rows)
    Q = torch.from_numpy(load_emb_rows(split, "s1", q_rows)).to(DEVICE, INFER_DTYPE)
    q_cc = d["s1_cc"][q_rows]
    by_cc = {int(c): torch.from_numpy(np.nonzero(q_cc == c)[0]).to(DEVICE) for c in np.unique(q_cc)}
    all_i, all_s = [], []
    qb = QUERY_BLOCK          # shrinks permanently if the GPU ever runs out of memory
    for src, offset, n_src in (("s2", 0, d["n_s2"]), ("s3", d["n_s2"], d["n_s3"])):
        best_s = torch.full((nq, K), -float("inf"), dtype=torch.float32, device=DEVICE)
        best_i = torch.full((nq, K), -1, dtype=torch.int64, device=DEVICE)
        for sh in range(n_shards(n_src)):
            E_np = np.load(shard_path(split, src, sh))
            start = sh * EMB_SHARD_ROWS
            c_cc = d["c_cc"][offset + start: offset + start + len(E_np)]
            for cc, q_idx in by_cc.items():
                cm = np.nonzero(c_cc == cc)[0]
                if len(cm) == 0:
                    continue
                Ec = torch.from_numpy(E_np[cm]).to(DEVICE, INFER_DTYPE)
                g_idx = torch.from_numpy(offset + start + cm).to(DEVICE)
                b = 0
                while b < len(q_idx):
                    qi = q_idx[b:b + qb]
                    try:
                        S = Q[qi] @ Ec.T
                        ts, ti = S.topk(min(K, S.shape[1]), dim=1)
                        cs = torch.cat([best_s[qi], ts.float()], dim=1)
                        ci = torch.cat([best_i[qi], g_idx[ti]], dim=1)
                        ns, pos = cs.topk(K, dim=1)
                        best_s[qi], best_i[qi] = ns, ci.gather(1, pos)
                        b += len(qi)
                    except RuntimeError as e:
                        if not is_oom(e) or qb == 1:
                            raise
                        S = ts = ti = cs = ci = None
                        free_gpu()
                        qb = max(1, qb // 2)
                        log(f"  OOM in retrieval, query block -> {qb}")
                    S = None
                del Ec, g_idx
            del E_np
        all_i.append(best_i); all_s.append(best_s)
    idx, sc = torch.cat(all_i, 1), torch.cat(all_s, 1)
    sc, order = sc.sort(dim=1, descending=True)
    idx = idx.gather(1, order)
    out = {"q_rows": q_rows, "idx": idx.cpu().numpy(), "score": sc.cpu().numpy()}
    del Q, idx, sc, order, all_i, all_s
    free_gpu()
    return out

def cached_retrieve(name, split, q_rows):
    path = f"{P['cand']}/{name}.npz"
    if os.path.exists(path):
        r = load_npz(path)
        if np.array_equal(r["q_rows"], np.asarray(q_rows)):
            log(f"{name}: loaded cached candidates")
            return r
        log(f"{name}: cached candidates are for different queries, recomputing")
    t0 = time.time()
    r = retrieve(split, q_rows)
    save_npz(path, **r)
    log(f"{name}: retrieved {len(q_rows):,} queries in {time.time()-t0:.0f}s")
    return r

# one pass over the train embeddings for both query sets (val scoring + CE training)
_q = np.concatenate([SPL["val_eval"], SPL["ce_rows"]])
_r = cached_retrieve("train_queries", "train", _q)
_nv = len(SPL["val_eval"])
CAND_VAL = {k: v[:_nv] for k, v in _r.items()}
CAND_CE = {k: v[_nv:] for k, v in _r.items()}
del _r

## 9. Recall@K (blocking ceiling) on validation

`edge recall` = share of true match links found in the top K; `all found` = share of non-singleton
entities whose full match set is in the top K; `oracle F0.5` = the best score a perfect reranker could
reach with those candidates (singletons included). The row at `K = RERANK_K` is the real ceiling
for everything downstream.

In [ ]:
# ---- Cell 9: recall report ----
def hit_matrix(cand, n_corpus=N_C_TR):
    keys = cand["q_rows"][:, None] * n_corpus + cand["idx"]
    return np.isin(keys, GT_KEYS) & (cand["idx"] >= 0)

def f05_from_counts(tp, n_pred, n_true):
    tp, n_pred, n_true = (np.asarray(x, dtype=np.float64) for x in (tp, n_pred, n_true))
    p = np.divide(tp, n_pred, out=np.zeros_like(tp), where=n_pred > 0)
    r = np.divide(tp, n_true, out=np.zeros_like(tp), where=n_true > 0)
    den = 0.25 * p + r
    f = np.divide(1.25 * p * r, den, out=np.zeros_like(tp), where=den > 0)
    return np.where(n_true == 0, (n_pred == 0).astype(np.float64), f)

# self-test against the problem statement's worked example (precision 2/3, recall 1 -> 0.714)
assert abs(f05_from_counts([2], [3], [2])[0] - 0.7142857142857143) < 1e-9
assert f05_from_counts([0], [0], [0])[0] == 1.0 and f05_from_counts([0], [1], [0])[0] == 0.0
assert f05_from_counts([0], [0], [2])[0] == 0.0

HIT_VAL = hit_matrix(CAND_VAL)
T_VAL = TRUE_COUNT[CAND_VAL["q_rows"]]
ns = T_VAL > 0
print(f"validation entities: {len(T_VAL):,} ({(~ns).sum():,} singletons)")
print(f"{'K':>4} | edge recall | all found | oracle F0.5")
for k in sorted({1, 3, 5, 10, 15, RERANK_K, 2 * K_PER_SOURCE}):
    found = HIT_VAL[:, :k].sum(1)
    print(f"{k:>4} |   {found.sum() / T_VAL.sum():.4f}    |  {(found[ns] == T_VAL[ns]).mean():.4f}   |"
          f"   {f05_from_counts(found, found, T_VAL).mean():.4f}{'   <- reranker input' if k == RERANK_K else ''}")

## 10. Cross-encoder training pairs

For each of the `N_CE_TRAIN_ENTITIES` training entities:
- **positives:** every ground-truth match (including ones retrieval missed, so the model also sees
  hard positives),
- **negatives:** the `NEG_HARD` highest-ranked retrieved non-matches plus `NEG_RANDOM` random other
  retrieved non-matches from the top `RERANK_K`. These are the confusable records the reranker will
  actually face at inference. Singleton entities contribute negatives only, which teaches "no match".

In [ ]:
# ---- Cell 10: build CE training pairs ----
pairs_path = f"{P['ce_data']}/pairs.parquet"
if os.path.exists(pairs_path):
    PAIRS = pd.read_parquet(pairs_path)
else:
    rng = np.random.default_rng(SEED + 1)
    hit = hit_matrix(CAND_CE)[:, :RERANK_K]
    top = CAND_CE["idx"][:, :RERANK_K]
    neg_a, neg_b = [], []
    for i, r in enumerate(CAND_CE["q_rows"]):
        negs = top[i][(~hit[i]) & (top[i] >= 0)]
        chosen = list(negs[:NEG_HARD])
        rest = negs[NEG_HARD:]
        if len(rest):
            chosen += list(rng.choice(rest, size=min(NEG_RANDOM, len(rest)), replace=False))
        neg_a += [r] * len(chosen); neg_b += chosen
    pos_mask = np.isin(GT["s1"], CAND_CE["q_rows"])
    PAIRS = pd.DataFrame({
        "a": np.concatenate([GT["s1"][pos_mask], np.array(neg_a, np.int64)]),
        "b": np.concatenate([GT["c"][pos_mask], np.array(neg_b, np.int64)]),
        "label": np.concatenate([np.ones(pos_mask.sum(), np.float32), np.zeros(len(neg_a), np.float32)]),
    })
    save_parquet(PAIRS, pairs_path)
print(f"CE pairs: {len(PAIRS):,} | positives {int(PAIRS.label.sum()):,} | negatives {int((PAIRS.label == 0).sum()):,}")
print("example positive:", TR["s1_text"][PAIRS.a.iloc[0]], " <-> ", TR["c_text"][PAIRS.b.iloc[0]])

## 11. Fine-tune `bge-reranker-v2-m3`

A plain PyTorch loop (no Trainer, so no library-version surprises) with:
- **mixed precision:** bf16 on L4/A100, fp16 + GradScaler on T4; fp32 master weights,
- **length-grouped batches:** shuffled mega-batches, sorted by length inside, so padding stays small,
- **AdamW:** linear warmup/decay, grad clipping at 1.0, binary cross-entropy on the model's single logit,
- **OOM guard:** a batch that runs out of memory is skipped (logged) instead of killing the run,
- **checkpoint to Drive every `CKPT_MINUTES`:** model + optimizer + scheduler + step. Re-running
  the cell resumes from the exact step. The finished model is saved to `ce_final/` in fp16 (~1.1 GB).

Time: roughly 1.7M pairs at batch 32 is ~53k steps. Expect a few hours on an L4 (the log prints the
real ETA).

In [ ]:
# ---- Cell 11: train the cross-encoder (resumable) ----
CE_DONE = f"{P['ce_final']}/DONE"
CKPT = f"{P['ce_ckpt']}/last.pt"

def epoch_batches(epoch, lens, bs):
    rng = np.random.default_rng(SEED + 100 + epoch)
    perm = rng.permutation(len(lens))
    mega = bs * 64
    batches = []
    for s in range(0, len(perm), mega):
        chunk = perm[s:s + mega]
        chunk = chunk[np.argsort(lens[chunk], kind="stable")]
        batches += [chunk[j:j + bs] for j in range(0, len(chunk), bs)]
    return [batches[i] for i in rng.permutation(len(batches))]

def save_ckpt(state):
    local = f"{LOCAL_TMP}/last.pt"
    torch.save(state, local)
    shutil.copy(local, _tmp(CKPT))
    os.replace(_tmp(CKPT), CKPT)
    os.remove(local)

if os.path.exists(CE_DONE):
    print("Cross-encoder already trained:", P["ce_final"])
else:
    # mmap=True: the ~7 GB checkpoint is paged in from disk instead of loaded into RAM all at once
    ck = torch.load(CKPT, map_location="cpu", weights_only=False, mmap=True) if os.path.exists(CKPT) else None
    bs_train = ck["batch"] if ck is not None else CE_BATCH     # resume with the batch size it started with
    a_txt = TR["s1_text"][PAIRS["a"].values]
    b_txt = TR["c_text"][PAIRS["b"].values]
    y_all = PAIRS["label"].values.astype(np.float32)
    lens = np.fromiter((len(x) + len(z) for x, z in zip(a_txt, b_txt)), np.int64, len(a_txt))
    n_batches = math.ceil(len(y_all) / bs_train)
    total_steps = n_batches * CE_EPOCHS
    warm = max(1, int(total_steps * CE_WARMUP_FRAC))

    ce_tok = AutoTokenizer.from_pretrained(RERANKER_MODEL)
    ce_model = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL, num_labels=1)
    ce_model = ce_model.float().to(DEVICE)
    ce_model.train()
    no_decay = ("bias", "norm", "LayerNorm")
    groups = [
        {"params": [p for n, p in ce_model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": CE_WEIGHT_DECAY},
        {"params": [p for n, p in ce_model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=CE_LR)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))
    use_scaler = DEVICE == "cuda" and AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    step = 0
    if ck is not None:
        assert ck["total_steps"] == total_steps, \
            "checkpoint was made from different training pairs; delete ce_ckpt/last.pt to restart training"
        ce_model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        if use_scaler and ck["scaler"]:
            scaler.load_state_dict(ck["scaler"])
        step = ck["step"]
        ck = None   # free the ~7 GB checkpoint copy in RAM
        gc.collect()
        log(f"Resumed cross-encoder training from step {step:,}/{total_steps:,}")
    log(f"Training {RERANKER_MODEL}: {len(y_all):,} pairs, batch {bs_train}, {total_steps:,} steps, "
        f"amp={AMP_DTYPE}, scaler={use_scaler}")

    cur_epoch, batches = -1, None
    t0, t_ckpt, losses, skipped, start_step = time.time(), time.time(), [], 0, step
    while step < total_steps:
        ep, b = divmod(step, n_batches)
        if ep != cur_epoch:
            batches, cur_epoch = epoch_batches(ep, lens, bs_train), ep
        bi = batches[b]
        enc = ce_tok(list(a_txt[bi]), list(b_txt[bi]), padding=True, truncation=True,
                     max_length=CE_MAX_LEN, return_tensors="pt").to(DEVICE)
        yb = torch.from_numpy(y_all[bi]).to(DEVICE)
        oom = False
        try:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
                logits = ce_model(**enc).logits.squeeze(-1)
            loss = F.binary_cross_entropy_with_logits(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(ce_model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
        except RuntimeError as e:
            if not is_oom(e):
                raise
            oom = True
        opt.zero_grad(set_to_none=True)
        logits = loss = None
        if oom:
            max_len = int(enc["input_ids"].shape[1])
            enc = yb = None
            free_gpu()
            if use_scaler:   # a half-finished step leaves the scaler mid-update; start a fresh one
                scaler = torch.amp.GradScaler("cuda", init_scale=scaler.get_scale(), enabled=True)
            skipped += 1
            log(f"  OOM on a batch of {len(bi)} (max len {max_len}), skipped ({skipped} so far)")
            if skipped > 200:
                raise RuntimeError("Too many OOM batches: lower CE_BATCH or CE_MAX_LEN in Cell 2")
        sched.step()
        step += 1

        if step % 500 == 0 or step == total_steps:
            rate = (step - start_step) / (time.time() - t0)
            log(f"  step {step:,}/{total_steps:,} | loss {np.mean(losses[-500:]) if losses else float('nan'):.4f} | "
                f"lr {sched.get_last_lr()[0]:.2e} | {rate:.2f} it/s | ETA {(total_steps - step) / rate / 60:.0f} min")
        if time.time() - t_ckpt > CKPT_MINUTES * 60 and step < total_steps:
            save_ckpt({"model": ce_model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                       "scaler": scaler.state_dict(), "step": step, "total_steps": total_steps, "batch": bs_train})
            t_ckpt = time.time()
            log(f"  checkpoint saved to Drive at step {step:,}")

    ce_model.eval()
    if DEVICE == "cuda":
        ce_model = ce_model.half()
    local_final = f"{LOCAL_TMP}/ce_final"
    ce_model.save_pretrained(local_final)
    ce_tok.save_pretrained(local_final)
    shutil.copytree(local_final, P["ce_final"], dirs_exist_ok=True)
    with open(CE_DONE, "w") as f:
        f.write(json.dumps({"steps": total_steps, "pairs": int(len(y_all)), "skipped_oom": skipped}))
    log(f"Cross-encoder saved to {P['ce_final']} (skipped OOM batches: {skipped})")
    del ce_model, opt, sched, scaler
    free_gpu()

## 12. Score validation candidates + tune the decision rule for macro F0.5

Decision rule (two thresholds, both tuned on validation):
- predict every candidate with score ≥ `t_all`,
- if none passes, still predict the single best candidate when its score ≥ `t_top` (`t_top ≤ t_all`).

The second rule matters under F0.5: a non-singleton with no prediction scores 0, while one confident
top-1 match usually scores well. The simple single-threshold result is printed alongside for comparison.

In [ ]:
# ---- Cell 12: validation scoring + threshold search ----
def load_reranker():
    tok = AutoTokenizer.from_pretrained(P["ce_final"])
    model = AutoModelForSequenceClassification.from_pretrained(P["ce_final"]).to(DEVICE).eval()
    model = model.half() if DEVICE == "cuda" else model.float()
    return tok, model

@torch.inference_mode()
def ce_score(a_list, b_list, tok, model, bs):
    lens = np.fromiter((len(x) + len(z) for x, z in zip(a_list, b_list)), np.int64, len(a_list))
    order = np.argsort(lens, kind="stable")[::-1]
    out = np.empty(len(a_list), np.float32)
    i = 0
    while i < len(order):
        idx = order[i:i + bs]
        try:
            enc = tok([a_list[j] for j in idx], [b_list[j] for j in idx], padding=True, truncation=True,
                      max_length=CE_MAX_LEN, return_tensors="pt").to(DEVICE)
            out[idx] = torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy()
            i += len(idx)
        except RuntimeError as e:
            if not is_oom(e) or bs == 1:
                raise
            enc = None
            free_gpu()
            bs = max(1, bs // 2)
            log(f"  OOM while scoring, batch -> {bs}")
    return out, bs

def score_candidates(tag, split, cand, K=RERANK_K):
    """CE-score the top-K candidates of every query. Returns [nq, K] scores (NaN = no candidate).
    Saved to Drive in chunks of SCORE_CHUNK pairs, so an interrupted run resumes."""
    d = DATA[split]
    top = cand["idx"][:, :K]
    qi, kj = np.nonzero(top >= 0)
    out_dir = f"{P['scores']}/{tag}"
    os.makedirs(out_dir, exist_ok=True)
    meta = {"n_pairs": int(len(qi)), "chunk": SCORE_CHUNK, "K": K,
            "ce_done": open(CE_DONE).read()}
    meta_p = f"{out_dir}/meta.json"
    if os.path.exists(meta_p) and json.load(open(meta_p)) != meta:
        log(f"  {tag}: stale scores (different candidates or model), rescoring")
        shutil.rmtree(out_dir); os.makedirs(out_dir)
    save_json(meta, meta_p)
    n_chunks = math.ceil(len(qi) / SCORE_CHUNK)
    todo = [c for c in range(n_chunks) if not os.path.exists(f"{out_dir}/chunk_{c:05d}.npy")]
    if todo:
        tok, model = load_reranker()
        bs, t0, done = SCORE_BATCH, time.time(), 0
        for c in todo:
            sl = slice(c * SCORE_CHUNK, (c + 1) * SCORE_CHUNK)
            a = d["s1_text"][cand["q_rows"][qi[sl]]]
            b = d["c_text"][top[qi[sl], kj[sl]]]
            s, bs = ce_score(list(a), list(b), tok, model, bs)
            save_npy(f"{out_dir}/chunk_{c:05d}.npy", s)
            done += len(s)
            rate = done / (time.time() - t0)
            left = len(qi) - c * SCORE_CHUNK - len(s)
            log(f"  {tag}: chunk {c+1}/{n_chunks} | {rate:,.0f} pairs/s | ETA {left / rate / 60:.0f} min")
        del model, tok
        free_gpu()
    flat = np.concatenate([np.load(f"{out_dir}/chunk_{c:05d}.npy") for c in range(n_chunks)]) \
        if n_chunks else np.zeros(0, np.float32)
    S = np.full(top.shape, np.nan, np.float32)
    S[qi, kj] = flat
    return S

def apply_rule(S, t_all, t_top):
    """Boolean [nq, K] prediction mask for the two-threshold rule. NaN = no candidate (never predicted)."""
    Sz = np.nan_to_num(S, nan=-1.0)
    pred = Sz >= t_all
    best = Sz.argmax(1)
    rows = np.arange(len(Sz))
    add_top = (~pred.any(1)) & (Sz[rows, best] >= t_top)
    pred[rows[add_top], best[add_top]] = True
    return pred

S_VAL = score_candidates("val_eval", "train", CAND_VAL)
H_VAL = HIT_VAL[:, :RERANK_K]

def macro_f05(pred):
    return f05_from_counts((pred & H_VAL).sum(1), pred.sum(1), T_VAL)

_Sz = np.nan_to_num(S_VAL, nan=-1.0)
_top_s = _Sz.max(1)
_top_hit = H_VAL[np.arange(len(_Sz)), _Sz.argmax(1)].astype(np.int64)
grid = np.round(np.arange(0.02, 0.991, 0.01), 2)
best, single = (-1.0, None, None), []
for t_all in grid:
    p_all = _Sz >= t_all
    tp_all, n_all = (p_all & H_VAL).sum(1), p_all.sum(1)
    single.append((float(t_all), f05_from_counts(tp_all, n_all, T_VAL).mean()))
    empty = n_all == 0
    for t_top in grid[grid <= t_all]:
        add = empty & (_top_s >= t_top)
        f = f05_from_counts(tp_all + add * _top_hit, n_all + add, T_VAL).mean()
        if f > best[0]:
            best = (float(f), float(t_all), float(t_top))
best_single = max(single, key=lambda x: x[1])
F_BEST, T_ALL, T_TOP = best
pred = apply_rule(S_VAL, T_ALL, T_TOP)
per = macro_f05(pred)
assert abs(per.mean() - F_BEST) < 1e-9, "fast threshold search disagrees with apply_rule"
print(f"single threshold      : t={best_single[0]:.2f}  macro F0.5 = {best_single[1]:.4f}")
print(f"two-threshold (chosen): t_all={T_ALL:.2f} t_top={T_TOP:.2f}  macro F0.5 = {F_BEST:.4f}")
print(f"   singletons     {per[T_VAL == 0].mean():.4f}  (n={int((T_VAL == 0).sum()):,})")
print(f"   non-singletons {per[T_VAL > 0].mean():.4f}  (n={int((T_VAL > 0).sum()):,})")
tp, npred = (pred & H_VAL).sum(), pred.sum()
print(f"   micro precision {tp / max(npred, 1):.4f} | micro recall {tp / T_VAL.sum():.4f}")
save_json({"t_all": T_ALL, "t_top": T_TOP, "val_macro_f05": float(F_BEST),
           "single_threshold": float(best_single[0]), "single_f05": float(best_single[1]),
           "rerank_k": RERANK_K, "n_val_entities": int(len(T_VAL))}, f"{WORK}/thresholds.json")
print("saved", f"{WORK}/thresholds.json")

## 13. Test set → `submission/matching_results.tsv` + `submission/candidate_pairs.tsv`

Same frozen embeddings, same country hard filter (France gets its own group from the test data), same
reranker, same thresholds as validation. `candidate_pairs.tsv` is exactly the top-`RERANK_K` set the
cross-encoder scores, so every predicted match is also a candidate. The files are checked against every
rule in the problem statement before the cell reports success.

In [ ]:
# ---- Cell 13: test inference + submission files ----
def write_id_lists(path, s1_ids, lists, col):
    tmp = _tmp(path)
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for sid, ids in zip(s1_ids, lists):
            f.write(f"{sid}\t{','.join(ids)}\n")
    os.replace(tmp, path)

def validate_submission(match_path, cand_path, d):
    problems = []
    valid_ids = set(d["c_ids"])
    s1_set = set(d["s1_ids"])
    tables = {}
    for path, col in ((match_path, "matched_entity_ids"), (cand_path, "candidate_entity_ids")):
        df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
        if list(df.columns) != ["source1_entity_id", col]:
            problems.append(f"{path}: bad header {list(df.columns)}")
        if df["source1_entity_id"].duplicated().any():
            problems.append(f"{path}: duplicate source1_entity_id rows")
        if set(df["source1_entity_id"]) != s1_set:
            problems.append(f"{path}: S1 ids differ from test_source1 ({len(df):,} rows vs {len(s1_set):,})")
        lists = {}
        for sid, v in zip(df["source1_entity_id"], df[col]):
            ids = [x for x in v.split(",") if x] if v else []
            if len(ids) != len(set(ids)):
                problems.append(f"{path}: duplicate ids for {sid}")
            bad = [x for x in ids if x not in valid_ids or not x.startswith(("S2-", "S3-"))]
            if bad:
                problems.append(f"{path}: unknown/non-S2/S3 ids for {sid}: {bad[:3]}")
            lists[sid] = set(ids)
        tables[col] = lists
    not_sub = sum(1 for s, m in tables["matched_entity_ids"].items()
                  if not m <= tables["candidate_entity_ids"].get(s, set()))
    if not_sub:
        problems.append(f"{not_sub} entities have matches outside their candidates")
    return problems[:20]

if not HAVE_TEST:
    print(f"No test files in {TEST_DIR}. Upload {TEST_FILES} there and Run all again; "
          "everything above is cached, so it jumps straight here.")
else:
    TE = DATA["test"]
    thr = json.load(open(f"{WORK}/thresholds.json"))
    all_rows = np.arange(len(TE["s1_ids"]))
    CAND_TEST = cached_retrieve("test_all", "test", all_rows)
    top = CAND_TEST["idx"][:, :RERANK_K]
    cand_lists = [[TE["c_ids"][j] for j in row if j >= 0] for row in top]
    cand_path = f"{P['submission']}/candidate_pairs.tsv"
    write_id_lists(cand_path, TE["s1_ids"], cand_lists, "candidate_entity_ids")
    n_c = np.array([len(x) for x in cand_lists])
    print(f"candidate_pairs.tsv: {len(cand_lists):,} rows, avg {n_c.mean():.1f} candidates, "
          f"{(n_c == 0).sum():,} with none")

    S_TEST = score_candidates("test_all", "test", CAND_TEST)
    pred = apply_rule(S_TEST, thr["t_all"], thr["t_top"])
    match_lists = [[TE["c_ids"][j] for j, p in zip(top[i], pred[i]) if p] for i in range(len(top))]
    match_path = f"{P['submission']}/matching_results.tsv"
    write_id_lists(match_path, TE["s1_ids"], match_lists, "matched_entity_ids")
    n_m = np.array([len(x) for x in match_lists])
    print(f"matching_results.tsv: {len(match_lists):,} rows | avg {n_m.mean():.2f} matches | "
          f"predicted singletons {(n_m == 0).mean():.2%} (train GT: 5.6%)")
    cc_names = np.array(TE["countries"])[TE["s1_cc"]]
    for c in TE["countries"]:
        m = cc_names == c
        print(f"   {c:>10}: {m.sum():,} entities, avg {n_m[m].mean():.2f} matches, "
              f"{(n_m[m] == 0).mean():.2%} empty")

    problems = validate_submission(match_path, cand_path, TE)
    if problems:
        print("VALIDATION PROBLEMS:"); [print("  -", p) for p in problems]
    else:
        print("\nPASS: both files follow every rule in the problem statement.")
        print("Upload to the portal:", match_path)